# Automization of Data Download

In [1]:
import time
import requests
import pandas as pd
import altair as alt

## Getting Markets

In [2]:
# Reading CSV with market info
market_receipt = pd.read_csv("other/market_list.csv")
market_receipt

,Name,Resolution,Category,Size (Vol),Resolution Date,Slug,Out Prefix
0,30-year mortgage rate below 6% by December 31?,"""No""",Economy,$38k Vol,"Dec 31, 2025",30-year-mortgage-rate-below-6-by-december-31,30-31
1,Will Christopher Waller dissent the next Fed D...,"""No""",Economy,$18k Vol,"Mar 18, 2026",will-waller-dissent-the-next-fed-decision,will-decision
2,Will Stephen Miran dissent the next Fed decision?,"""Yes""",Economy,$47k Vol,"Mar 18, 2026",will-stephen-miran-dissent-the-next-fed-decision,will-decision
3,Negative GDP growth in Q4 2025?,"""No""",Economy,$36k Vol,"Jan 29, 2026",negative-gdp-growth-in-q4-2025-295,negative-295
4,US bank failure by February 28?,"""Yes""",Economy,$78k Vol,"Feb 28, 2026",us-bank-failure-by-february-28,us-28
5,BLS delays another CPI release before 2027?,"""Yes""",Economy,$40k Vol,"Feb 5, 2026",bls-delays-another-cpi-release-before-2027,bls-2027
6,SpaceX and xAI merger officially announced by ...,"""Yes""",Economy,$105k Vol,"Feb 3, 2026",spacex-and-xai-merger-offcially-announced-by-j...,spacex-30
7,Nothing Ever Happens: February,"""Something""",Geopolitics,$86k Vol.,"Feb 28, 2026",nothing-ever-happens-february-152,nothing-152
8,Fact Check: is LA U-Haul attack perp a U.S. Ci...,"""No""",Geopolitics,$26k Vol.,"Mar 1, 2026",fact-check-is-la-u-haul-attack-perp-a-us-citizen,fact-citizen
9,Will Marco Rubio visit Israel by March 2?,"""No""",Geopolitics,$101k Vol,"Mar 2, 2026",will-marco-rubio-visit-israel-by-march-2,will-2


In [3]:
# Read Function
GAMMA = "https://gamma-api.polymarket.com"
DATA  = "https://data-api.polymarket.com"

def get_market(market_id: str = None, slug: str = None) -> dict:
    if (market_id is None) == (slug is None):
        raise ValueError("Provide either a market_id or slug.")

    if market_id:
        r = requests.get(f"{GAMMA}/markets/{market_id}", timeout=30)
    else:
        r = requests.get(f"{GAMMA}/markets/slug/{slug}", timeout=30)

    r.raise_for_status()
    return r.json()


# def fetch_all_trades(condition_id: str, limit: int = 500):
#     all_trades = []
#     before = None
#     while True:
#         params = {
#             "market": condition_id,
#             "limit": limit,
#         }
#         if before:
#             params["before"] = before
#         r = requests.get(f"{DATA}/trades", params=params, timeout=30)
#         r.raise_for_status()
#         trades = r.json()
#         if not trades:
#             break
#         all_trades.extend(trades)
#         print(
#             f"Fetched {len(all_trades)} trades so far... | "
#             f"batch newest: {trades[0]['timestamp']} | "
#             f"batch oldest: {trades[-1]['timestamp']} | "
#             f"next before: {before}"
#         )
#         before = trades[-1]["timestamp"] - 1
#         if len(trades) < limit:
#             break
#         time.sleep(0.05)
#     return all_trades

def fetch_all_trades(condition_id: str, limit: int = 500, max_offset: int = 3000):
    all_trades = []
    offset = 0

    while offset <= max_offset:
        params = {
            "market": condition_id,
            "limit": limit,
            "offset": offset,
            "takerOnly": True,
        }

        r = requests.get(f"{DATA}/trades", params=params, timeout=30)

        # clean stop at cap
        if r.status_code == 400:
            print(f"Hit cap at offset={offset}. Returning {len(all_trades)} trades.")
            break

        r.raise_for_status()
        batch = r.json()

        if not batch:
            break

        all_trades.extend(batch)
        print(f"Fetched {len(all_trades)} trades so far... (offset={offset})")

        if len(batch) < limit:
            break

        offset += limit
        time.sleep(0.02)

    return all_trades


def download_market_trades(market_id: str = None,
                           slug: str = None,
                           out_prefix: str = "polymarket"):

    market = get_market(market_id=market_id, slug=slug)
    condition_id = market["conditionId"]

    trades = fetch_all_trades(condition_id)
    trades_df = pd.DataFrame(trades)

    filename = f"market_csvs/{out_prefix}_trades.csv"
    trades_df.to_csv(filename, index=False)

    print(f"Saved {len(trades_df)} trades to {filename}")

    return trades_df

In [4]:
# Downloading all markets

for _, row in market_receipt.iterrows():
    s = row["Slug"]
    op = row["Out Prefix"]
    download_market_trades(slug=s, out_prefix=op)

Fetched 336 trades so far... (offset=0)
Saved 336 trades to market_csvs/30-31_trades.csv
Fetched 287 trades so far... (offset=0)
Saved 287 trades to market_csvs/will-decision_trades.csv
Fetched 500 trades so far... (offset=0)
Fetched 558 trades so far... (offset=500)
Saved 558 trades to market_csvs/will-decision_trades.csv
Fetched 454 trades so far... (offset=0)
Saved 454 trades to market_csvs/negative-295_trades.csv
Fetched 411 trades so far... (offset=0)
Saved 411 trades to market_csvs/us-28_trades.csv
Fetched 197 trades so far... (offset=0)
Saved 197 trades to market_csvs/bls-2027_trades.csv
Fetched 466 trades so far... (offset=0)
Saved 466 trades to market_csvs/spacex-30_trades.csv
Fetched 500 trades so far... (offset=0)
Fetched 952 trades so far... (offset=500)
Saved 952 trades to market_csvs/nothing-152_trades.csv
Fetched 467 trades so far... (offset=0)
Saved 467 trades to market_csvs/fact-citizen_trades.csv
Fetched 500 trades so far... (offset=0)
Fetched 854 trades so far... (of